In [ ]:
!pip install nltk beautifulsoup4 pymupdf


In [ ]:
## All the libraries:
import os
import fitz  
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [8]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\surya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\surya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\surya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [10]:
def extract_text_from_pdf(filepath):
    doc = fitz.open(filepath)
    return "\n".join([page.get_text() for page in doc])

In [11]:
def extract_text_from_html(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f, 'html.parser')
        return soup.get_text(separator="\n", strip=True)

In [16]:

    
def is_misnamed_pdf(filepath):
    try:
        with open(filepath, 'rb') as f:
            header = f.read(5)
            return header == b'%PDF-'
    except:
        return False


In [17]:
def get_clean_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.pdf' or is_misnamed_pdf(filepath):
        return extract_text_from_pdf(filepath)
    elif ext == '.html' or ext == '.htm':
        return extract_text_from_html(filepath)
    else:
        return ""

In [14]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    filtered_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    return " ".join(filtered_tokens)


In [15]:
def process_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        if not os.path.isfile(filepath):
            continue
        try:
            raw_text = get_clean_text(filepath)
            clean_text = preprocess_text(raw_text)
            out_file = os.path.join(output_dir, f"{filename}.txt")
            with open(out_file, "w", encoding="utf-8") as f:
                f.write(clean_text)
            print(f"Processed: {filename}")
        except Exception as e:
            print(f"Failed to process {filename}: {e}")


In [ ]:
input_folder = r""
output_folder = r""
process_folder(input_folder, output_folder)